In [ ]:
print("hello")
import sys
print(sys.version)


In [ ]:
!uv add langchain-google-genai langchain langchain-ollama langchain-community pypdf requests langchain-text-splitters
!uv add langchain
!uv add langchain-community
%pip install langchain-chroma langchain-huggingface sentence_transformers --quiet
!uv add pypdf

# %pip list

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv
# from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import AIMessage
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from pprint import pprint
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
load_dotenv()

In [ ]:
import os
import json
import hashlib

TRACKED_FILES_PATH = "tracked_files.json"

def get_file_hash(filepath):
    """Generate MD5 hash of a file to detect changes."""
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def load_tracked_files():
    if os.path.exists(TRACKED_FILES_PATH):
        with open(TRACKED_FILES_PATH, "r") as f:
            return json.load(f)
    return {}

def save_tracked_files(tracked):
    with open(TRACKED_FILES_PATH, "w") as f:
        json.dump(tracked, f, indent=2)

def get_new_files(files_dir="files"):
    tracked = load_tracked_files()
    new_files = []

    for filename in os.listdir(files_dir):
        if not filename.endswith(".pdf"):
            continue
        filepath = os.path.join(files_dir, filename)
        file_hash = get_file_hash(filepath)

        if filename not in tracked or tracked[filename] != file_hash:
            new_files.append(filepath)

    return new_files, tracked

# --- Main logic ---
new_files, tracked = get_new_files("files")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# Embed and store
vectorstore = Chroma(
    collection_name="web_rag",
    embedding_function=embedding_model,
    persist_directory="./rag_db"
)

if not new_files:
    print("No new files detected. Skipping embedding.")
else:
    print(f"Found {len(new_files)} new/modified file(s): {new_files}")

    # Load only new files
    loader = PyPDFDirectoryLoader("files")
    all_docs = loader.load()

    # Filter docs belonging to new files only
    new_docs = [doc for doc in all_docs if doc.metadata.get("source") in new_files]

    # Split
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(new_docs)
    
    document_ids = vectorstore.add_documents(documents=chunks)
    print(f"Added {len(document_ids)} chunks to vectorstore.")

    # Update tracker with newly processed files
    for filepath in new_files:
        filename = os.path.basename(filepath)
        tracked[filename] = get_file_hash(filepath)
    save_tracked_files(tracked)
    print("Tracker updated.")

In [ ]:
# loader  = PyPDFDirectoryLoader("files")
# docs = loader.load()

In [ ]:


# splitter = RecursiveCharacterTextSplitter(
#     chunk_size = 1000,
#     chunk_overlap = 200
# )
# chunks = splitter.split_documents(docs)

In [ ]:
# pprint(chunks)

In [ ]:
# from langchain_ollama import OllamaEmbeddings
# embeddings = OllamaEmbeddings(
#     model="nomic-embed-text"
# )

In [ ]:
# vectorstore = Chroma(
#     collection_name= "web_rag",
#     embedding_function=embedding_model,
#     persist_directory="./rag_db"
# )
# document_ids = vectorstore.add_documents(documents=chunks)

In [ ]:
# vectorstore.get(limit = 10, include = ['documents'])

# Main Code


In [ ]:

reasoning_model = ChatOllama(
    model="qwen3:14b",
    temperature=0,
    verbose=True,
    reasoning=False
)

In [ ]:
vision_llm   = ChatOllama(model="llava:7b",  temperature=0)
parser = StrOutputParser()

In [ ]:
!uv add langchain-tavily

In [ ]:
from langchain.tools import tool
import requests
import os

# Helper


In [ ]:
!uv add opencv-python
import cv2
import base64
from langchain_core.messages import HumanMessage
def encode_image(source) -> str:
    """Convert file path or OpenCV frame to base64."""
    if isinstance(source, str):
        frame = cv2.imread(source)
        if frame is None:
            raise FileNotFoundError(f"Image not found: {source}")
    else:
        frame = source

    frame = cv2.resize(frame, (640, 480))
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    _, buf = cv2.imencode(".jpg", frame_rgb, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return base64.b64encode(buf).decode("utf-8")

def ask_vision(image_source, prompt: str) -> str:
    b64 = encode_image(image_source)
    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}
    ])
    return parser.invoke(vision_llm.invoke([msg]))


# Tools

In [ ]:
@tool
def analyze_image(image_path: str, question: str) -> str:
    """
    Analyze an image file using the vision model.
    Use when the user provides an image path and wants to understand its contents.
    Args:
        image_path: Path to the image file
        question: What to look for or ask about the image
    """
    try:
        return ask_vision(image_path, question)
    except Exception as e:
        return f"Error analyzing image: {str(e)}"

In [ ]:
@tool
def capture_webcam(question: str, camera_index: int = 0) -> str:
    """
    Capture a frame from the webcam and analyze it.
    Use when the user wants to analyze what the camera currently sees or what the AI agent can see.
    Args:
        question: What to analyze in the captured frame
        camera_index: Camera device index (default 0)
    """
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        return "Error: Could not open webcam."

    # Warm up camera
    for _ in range(5):
        cap.read()

    ret, frame = cap.read()
    cap.release()

    if not ret:
        return "Error: Could not capture frame."

    result = ask_vision(frame, question)
    return f"[Webcam capture analyzed]\n{result}"


In [ ]:
@tool
def detect_objects(image_path: str) -> str:
    """
    Detect and list all objects in an image with their locations.
    Use when you need a structured inventory of objects in a scene.
    Args:
        image_path: Path to the image file
    """
    prompt = """List all objects you can detect in this image.
For each object provide:
- Object name
- Approximate location (top-left, center, bottom-right, etc.)
- Confidence (high/medium/low)

Format as a clean bullet list."""
    try:
        return ask_vision(image_path, prompt)
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
@tool
def compare_images(image_path_1: str, image_path_2: str, aspect: str = "general") -> str:
    """
    Compare two images and describe differences/similarities.
    Use when user wants to compare two images.
    Args:
        image_path_1: Path to first image
        image_path_2: Path to second image
        aspect: What to compare - 'general', 'objects', 'colors', 'scene', etc.
    """
    try:
        desc1 = ask_vision(image_path_1, f"Describe this image focusing on: {aspect}")
        desc2 = ask_vision(image_path_2, f"Describe this image focusing on: {aspect}")

        comparison_prompt = f"""Compare these two image descriptions and highlight key differences and similarities.
Focus on: {aspect}

Image 1: {desc1}

Image 2: {desc2}

Provide a structured comparison."""
        msgs = [HumanMessage(content=comparison_prompt)]
        return parser.invoke(reasoning_model.invoke(msgs))
    except Exception as e:
        return f"Error comparing images: {str(e)}"

In [ ]:
@tool
def reason_deeply(context: str, question: str) -> str:
    """
    Apply deep reasoning to a question given some context.
    Use for complex analysis, multi-step reasoning, or when you need to 
    draw conclusions from visual descriptions already gathered.
    Args:
        context: Background information or visual descriptions
        question: The question to reason about
    """
    prompt = f"""You are an expert analyst. Given the following context, 
reason carefully and thoroughly to answer the question.

Context:
{context}

Question:
{question}

Provide a detailed, well-reasoned answer with clear logic."""
    msgs = [HumanMessage(content=prompt)]
    return parser.invoke(reasoning_model.invoke(msgs))

In [ ]:
@tool
def read_text_in_image(image_path: str) -> str:
    """
    Extract and read all text visible in an image (OCR).
    Use when the user wants to read text from a photo, screenshot, or document image.
    Args:
        image_path: Path to the image containing text
    """
    prompt = """Extract ALL text visible in this image. 
Return it exactly as it appears, preserving formatting where possible.
If it's a document, preserve the structure (headings, paragraphs, lists, tables).
If no text is found, say 'No text detected.'"""
    try:
        return ask_vision(image_path, prompt)
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
def retrieve_context(query: str, k: int = 5):
  retrieved_docs = vectorstore.similarity_search(query, k=k)

  context = "\n\n".join(
      [
          f"Document: {d.metadata.get('source','unknown')}\n"
          f"Page: {d.metadata.get('page','?')}\n"
          f"Content: {d.page_content}"
          for d in retrieved_docs
      ]
  )

  return context, retrieved_docs

In [ ]:


tavily_api_key = os.getenv('TAVILY_API_KEY')


from langchain_tavily import TavilySearch

tavily = TavilySearch(
    max_results=20,
    search_depth="advanced",
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)
@tool
def search_web(query: str)->str:
    """Search the internet for current information."""
    print("\nCalling search_web tool")
    print(f"Query: {query}\n")

    results = tavily.invoke(query)
    return results


In [ ]:

from langchain_core.messages import HumanMessage
@tool
def ask_rag(query: str):
    """Search for Context in Files."""
    print(f"\nCalling ask_rag tool")
    context, retrieved_docs = retrieve_context(query)
    prompt = f"""
    You are reading retrieved context from internal documents.

    If the answer exists in the context, answer the question using it.

    If the context does not contain the answer, reply exactly with:
    NO_CONTEXT_FOUND

    Context:
    {context}

    Question:
    {query}
    """
    response = reasoning_model.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:

@tool
def get_job(skill:str, location:str)->list:
	"""Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
	print(f"\nCalling search_jobs tool")
	print(f"Searching jobs for: {skill} in {location}")
	url = "https://jsearch.p.rapidapi.com/search"

	querystring = {"query":f"{skill} in {location}","page":"1","num_pages":"1","country":f"{location}","date_posted":"all","employment_types":"INTERN"}

	headers = {
		"x-rapidapi-key": os.getenv("rapid_api_key"),
		"x-rapidapi-host": "jsearch.p.rapidapi.com"
	}

	response = requests.get(url, headers=headers, params=querystring)

	data = (response.json())
	jobs = data.get("data",[])

	print(f"found {len(jobs)} jobs")

	result = []
	for job in jobs:
		result.append({
			"Job Title": job.get("job_title",""),
			"employer name": job.get("employer_name",""),
			"apply link": job.get("job_apply_link",""),
			"location":job.get("job_location","")
		})
	return result


In [ ]:

@tool
def news_search(query:str) -> list :
    """Search for Current NEWS using NewsSearch API from RapidAPI."""
    print(f"\nCalling search news tool")
    url = "https://real-time-news-data.p.rapidapi.com/search"

    querystring = {"query":f"{query}","limit":"10","time_published":"anytime","country":"US","lang":"en"}

    headers = {
        "x-rapidapi-key": os.getenv("rapid_api_key"),
        "x-rapidapi-host": "real-time-news-data.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data =  response.json()
    news = data.get("data",[])
    print(f"found {len(news)} news")
    result = []
    for cur in news:
        result.append({
            "Article Title": cur.get("title",""),
            "Article Link" : cur.get("link",""),
            "Snippet": cur.get("snippet",""),
            "source": cur.get("source_name", ""),
            "source url": cur.get("source_url", ""),

        })
    return result


In [ ]:
# semantic scholar tool
@tool
def search_papers(query: str) -> str:
    """Search Semantic Scholar for research papers related to a topic."""
    
    print("\nCalling search_papers tool")
    print(f"Searching papers for: {query}\n")

    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    params = {
        "query": query,
        "limit": 5,
        "fields": "title,authors,year,abstract,url"
    }

    response = requests.get(url, params=params)
    data = response.json()

    papers = data.get("data", [])

    results = []

    for paper in papers:
        authors = ", ".join([a["name"] for a in paper.get("authors", [])])

        results.append(
            f"Title: {paper.get('title','')}\n"
            f"Authors: {authors}\n"
            f"Year: {paper.get('year','')}\n"
            f"Abstract: {paper.get('abstract','')}\n"
            f"URL: {paper.get('url','')}\n"
        )

    return "\n\n".join(results)

In [ ]:
#Hacker News Tool

@tool
def hackernews_search(topic: str) -> str:
    """Search Hacker News discussions related to a topic."""

    print("\nCalling hackernews_search tool")
    print(f"Topic: {topic}\n")

    base_url = "https://hacker-news.firebaseio.com/v0"

    # get top story ids
    ids = requests.get(f"{base_url}/topstories.json").json()[:50]

    results = []

    for story_id in ids:
        item = requests.get(f"{base_url}/item/{story_id}.json").json()

        if not item:
            continue

        title = item.get("title", "")
        url = item.get("url", "")
        score = item.get("score", 0)

        # crude relevance filter
        if topic.lower() in title.lower():
            results.append(
                f"Title: {title}\n"
                f"Score: {score}\n"
                f"URL: {url}\n"
            )

        if len(results) >= 5:
            break

    if not results:
        return "No relevant Hacker News discussions found."

    return "\n\n".join(results)

In [ ]:
#wikipedia seach tool
@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia and return a summary."""

    print("\nCalling wikipedia_search tool")
    print(f"Query: {query}\n")

    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + query

    response = requests.get(url)

    if response.status_code != 200:
        return "No Wikipedia article found."

    data = response.json()

    return f"""
Title: {data.get("title")}

Summary:
{data.get("extract")}

URL:
{data.get("content_urls", {}).get("desktop", {}).get("page")}
"""

In [ ]:
@tool
def weather_search(city: str) -> str:
    """Get current weather for a city."""

    print("\nCalling weather_search tool")

    api_key = os.getenv("WEATHER_API_KEY")

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

    response = requests.get(url)

    if response.status_code != 200:
        return f"Weather API error: {response.status_code}"

    data = response.json()

    # check if API returned an error
    if "weather" not in data:
        return f"Weather data not available. API response: {data}"

    weather = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    return f"""
City: {city}
Temperature: {temp} °C
Weather: {weather}
Humidity: {humidity} %
"""

In [ ]:
@tool
def location_search(place: str) -> str:
    """Find location coordinates and details."""

    print("\nCalling location_search tool")
    print(f"Searching location: {place}")

    url = "https://nominatim.openstreetmap.org/search"

    params = {
        "q": place,
        "format": "json",
        "limit": 1
    }

    headers = {
        "User-Agent": "Onix-AI-Agent"
    }

    try:
        response = requests.get(url, params=params, headers=headers)

        if response.status_code != 200:
            return f"Location API error: {response.status_code}"

        if not response.text:
            return "Location API returned empty response."

        data = response.json()

        if not data:
            return "Location not found."

        result = data[0]

        return f"""
Place: {result['display_name']}
Latitude: {result['lat']}
Longitude: {result['lon']}
"""

    except Exception as e:
        return f"Location tool failed: {str(e)}"

# Main

In [ ]:
from langchain.agents import create_agent
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model
from pprint import pprint
import requests

In [ ]:
!uv add langchain-ollama

In [ ]:
# model = init_chat_model(
#     model = "gemini-2.5-flash",
#     model_provider="google_genai",
#     api_key = api_key
# )

In [ ]:
# from datetime import datetime
# from zoneinfo import ZoneInfo

In [ ]:
!uv add langgraph

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver


In [ ]:
checkpointer = InMemorySaver()

In [ ]:
system_prompt = """
You are Onix, an advanced AI research assistant built to deliver accurate, 
well-verified, and comprehensive answers. If anyone asks your name, always 
respond that your name is Onix.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AVAILABLE TOOLS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ask_rag
  Search internal documents, files, and the private knowledge base.
  Always call this first before using any external tool.

search_web
  Search the internet for general information, facts, and verification.
  Use to enrich or confirm information from other sources.

get_job
  Find real, current job listings based on skills, role, or location.
  Use when the user asks about careers, jobs, or hiring.

news_search
  Retrieve current news articles and recent events.
  Use for anything time-sensitive or trending.

search_papers
  Search peer-reviewed research papers and scientific publications.
  Use for academic, technical, or evidence-based questions.

hackernews_search
  Search technology discussions, startup news, and developer topics on Hacker News.
  Use for cutting-edge tech trends and community perspectives.

wikipedia_search
  Retrieve structured factual information from Wikipedia.
  Use for definitions, historical context, and established facts.

weather_search
  Get real-time weather data for any location.
  Use when the user asks about weather, forecasts, or climate conditions.

location_search
  Retrieve geographic information, coordinates, and place details.
  Use for map-related queries, distance, or location context.

analyze_image
  Analyze an image file and describe its contents using vision AI.
  Use when the user provides an image path and wants visual understanding.

capture_webcam
  Capture a live frame from the webcam and analyze it.
  Use when the user wants real-time visual awareness from camera input or what the ai can see.

detect_objects
  Detect and list all objects in an image with their locations.
  Use when a structured inventory of visual elements is needed.

compare_images
  Compare two images and highlight differences and similarities.
  Use when the user wants a side-by-side visual comparison.

read_text_in_image
  Extract and read all visible text from an image (OCR).
  Use for screenshots, documents, receipts, signs, or any image with text.

reason_deeply
  Apply structured multi-step reasoning over gathered context.
  Use to draw conclusions, synthesize findings, or answer complex questions
  after collecting information from other tools.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REASONING PROCESS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Follow this decision process for every query:

Step 1 — Check Internal Knowledge
  Always call ask_rag first to check whether the answer exists in internal 
  documents. If strong, relevant context is found, use it as the primary source.

Step 2 — Enrich or Verify Externally
  After ask_rag, use search_web, wikipedia_search, news_search, or 
  search_papers to verify, expand, or update the internal findings.
  Do not skip this step for factual or time-sensitive topics.

Step 3 — Detect Query Type and Select Specialized Tools
  Match the user's intent to the right tool:
  - Image or visual input detected    → analyze_image, detect_objects, 
                                        read_text_in_image, compare_images
  - Live camera requested             → capture_webcam
  - Job or career query               → get_job
  - Current events or breaking news   → news_search
  - Academic or scientific question   → search_papers
  - Tech discussion or startup topic  → hackernews_search
  - Historical or definitional query  → wikipedia_search
  - Weather or forecast query         → weather_search
  - Location or geographic query      → location_search
  - Complex multi-source reasoning    → reason_deeply

Step 4 — Cross-Verify When Stakes Are High
  For important claims, run at least two independent tools and compare results.
  Note any contradictions and explain them clearly to the user.

Step 5 — Synthesize and Respond
  Combine all gathered information into a single, clear, well-structured answer.
  Always call reason_deeply when synthesizing findings from three or more tools.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEHAVIORAL RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

- Always introduce yourself as Onix if asked for your name.
- Never guess when a tool can give a better answer. Use the tool.
- Call tools multiple times if the first result is insufficient or ambiguous.
- Prefer internal knowledge from ask_rag when it is relevant and recent.
- Never fabricate facts, citations, job listings, or paper references.
- If two sources conflict, acknowledge both and explain the discrepancy.
- For visual tasks, always describe what you see before drawing conclusions.
- Keep responses focused. Do not pad answers with unnecessary repetition.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

- Write in clean, readable plain text only.
- Do not use markdown symbols such as **, ##, or bullet dashes.
- Use short paragraphs separated by blank lines for readability.
- Lead with the most important finding, then add supporting detail.
- Cite which tool or source each key piece of information came from.
- End complex answers with a brief summary of the key takeaway.
"""

In [ ]:

agent = create_agent(
    model=reasoning_model,
    tools=[ ask_rag,
    search_web,
    get_job,
    news_search,
    search_papers,
    hackernews_search,
    wikipedia_search,
    weather_search,
    location_search,
    analyze_image,
    capture_webcam,
    detect_objects,
    compare_images,
    read_text_in_image,
    reason_deeply,],
    system_prompt=system_prompt,
    debug=True,
    
)


In [ ]:
config = {"configurable": {"thread_id": "1"}}

In [ ]:
# def ask(query):
#     response = agent.invoke({
#             "messages":[{
#                 "role":"user", "content":query
#             }]
#         },config=config)
#     return response

In [ ]:
import langchain_ollama
print(langchain_ollama.__version__)  # must be 0.3.x or higher

In [ ]:
!uv add kokoro soundfile sounddevice --quiet

# Agent Calling

In [ ]:
import os
from kokoro import KPipeline
import sounddevice as sd
import numpy as np

pipeline = KPipeline(lang_code='a')

def voice_out(text: str, voice: str = 'af_heart', speed: float = 1.0):
    audio_chunks = []
    for _, _, audio in pipeline(text, voice=voice, speed=speed):
        audio_chunks.append(audio)
    if audio_chunks:
        full_audio = np.concatenate(audio_chunks)
        sd.play(full_audio, samplerate=24000)
        sd.wait()


In [ ]:
import traceback
def run_agent():
    print("Onix is ready.")
    print("Vision model : llava:7b")
    print("Reasoning    : qwen3:14b")
    print("Type 'quit' to exit.\n")

    chat_history = []   # manual rolling window memory

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break

        if not user_input:
            continue

        try:
            # Build full message list: history + new user message
            messages = chat_history + [{"role": "user", "content": user_input}]

            result = agent.invoke({"messages": messages})

            # print(result)
            answer = result["messages"][-1].content
            print(f"\nOnix: {answer}\n")
            voice_out(answer)

            print("-" * 60)

            # Append to rolling history
            chat_history.append({"role": "user",      "content": user_input})
            chat_history.append({"role": "assistant", "content": answer})

            # Keep last 10 turns (20 messages) to avoid context overflow
            if len(chat_history) > 20:
                chat_history = chat_history[-20:]


        except Exception as e:
            print("🔥 FULL ERROR TRACE:")
            traceback.print_exc()


if __name__ == "__main__":
    run_agent()

Onix is ready.
Vision model : llava:7b
Reasoning    : qwen3:14b
Type 'quit' to exit.

[values] {'messages': [HumanMessage(content='analyze this image "C:\\Users\\SATYAM PRAKASH\\Downloads\\IMG_20260423_132431403_HDR.jpg"', additional_kwargs={}, response_metadata={}, id='4758dab1-e7d9-4d5a-8405-9c2c58623385')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-05-03T06:15:35.7898137Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22959264000, 'load_duration': 8271979400, 'prompt_eval_count': 2288, 'prompt_eval_duration': 4570292800, 'eval_count': 67, 'eval_duration': 10027788400, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019dec7a-04b2-7800-9978-47aa6766fa13-0', tool_calls=[{'name': 'analyze_image', 'args': {'image_path': 'C:\\Users\\SATYAM PRAKASH\\Downloads\\IMG_20260423_132431403_HDR.jpg', 'question': 'What is in this image?'}, 'id': '58aa966c-